# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides users through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata.to_json()
print("Dataset Title:", metadata['name'])
print("Description:", metadata['description'])
print("Identifier:", metadata['identifier'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each entity (record set, field, column) is referenced using its `@id` field per the Croissant schema.

In [ ]:
# List available record sets by @id
record_sets = []

if 'recordSet' in metadata and metadata['recordSet']:
    for rec in metadata['recordSet']:
        if isinstance(rec, dict) and '@id' in rec:
            record_sets.append(rec['@id'])
        elif isinstance(rec, str):
            record_sets.append(rec)
else:
    # In FAIR^2, the recordSet might be empty in top-level metadata, but could be present in distributions
    print("No record sets found directly in metadata['recordSet']. Checking distributions...")
    # Attempt to retrieve record set IDs from the dataset's distribution, if available.
    distributions = metadata.get('distribution', [])
    if distributions:
        for dist in distributions:
            if isinstance(dist, dict) and '@id' in dist:
                print(f"Distribution @id: {dist['@id']}")
    else:
        print("No distributions found either.")

if record_sets:
    print("Record Sets (@id):", record_sets)
else:
    print("No record sets discovered. Consult package schema for location of tabular data.")

In [ ]:
# For demonstration, let's enumerate the fields for each record set found
for record_set_id in record_sets:
    print(f"\nInspecting record set: {record_set_id}")
    try:
        recset_meta = dataset.metadata.record_set(record_set_id)
        if hasattr(recset_meta, 'fields'):
            field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in recset_meta.fields]
            print(f"Fields (@id): {field_ids}")
        else:
            print("No fields found in this record set.")
    except Exception as e:
        print("Error loading fields for:", record_set_id, e)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}.")
    print(f"Columns (@id): {list(df.columns)}\n")
# For exploration, choose first record set (if exists)
if record_sets:
    primary_record_set = record_sets[0]
    print("Preview of primary DataFrame:")
    display(dataframes[primary_record_set].head())
else:
    print("No DataFrames to show.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section uses field `@id` for all column access.

In [ ]:
# Pick a numeric field @id for analysis (example: 'Age')
numeric_field_id = None
search_terms = ['age', 'Age', 'schema:age', 'dv:age']
for col in dataframes[primary_record_set].columns:
    if any(term in col for term in search_terms):
        numeric_field_id = col
        break

if numeric_field_id:
    # Filter records with age > threshold
    threshold = 50
    filtered_df = dataframes[primary_record_set][dataframes[primary_record_set][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (example: 'Sex')
    group_field_id = None
    group_search_terms = ['sex', 'Sex', 'schema:sex', 'dv:sex']
    for col in dataframes[primary_record_set].columns:
        if any(term in col for term in group_search_terms):
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name='mean_' + numeric_field_id)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields using their `@id`.

Here, we'll plot the age distribution and, if possible, compare it by sex.

In [ ]:
# Visualization of numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[primary_record_set][numeric_field_id], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[primary_record_set])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated procedural steps to load, explore, and visualize the FAIR² dataset using the `mlcroissant` package and referenced all data entities by their `@id` for reproducibility.

Key findings and practical steps:
- Loaded dataset metadata and records using Croissant schema URL.
- Identified record sets and fields using their `@id`s.
- Performed basic exploratory analysis and normalization of numeric fields.
- Visualized distributions and grouped summaries using consistent schema referencing.

Further exploration and domain-specific analysis can be conducted according to the clinical use cases outlined in the dataset metadata.